# Location Marginal Pricing - Initial GNN

In [ ]:
from functools import lru_cache
import time

from loguru import logger
import mlflow
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch_geometric.nn import SAGEConv

from lmp_model.config import PROCESSED_DATA_DIR
from lmp_model.dataset import FEATURES, SYSTEM_FEATURES, WEATHER_FEATURES, load_cache, load_nodes
from lmp_model.features import fit_scalers, prepare
from lmp_model.graph import build_edges, edge_index
from lmp_model.splits import split_masks, walk_forward_folds

folds = walk_forward_folds()
nodes = load_nodes()
pnode_ids = nodes.index
data = load_cache()
masks = split_masks(data["times"], folds[0])
scalers = fit_scalers(data, masks["train"])
blocks = {name: prepare(data, mk, scalers) for name, mk in masks.items()}
{name: int(m.sum()) for name, m in masks.items()}

In [ ]:
torch.set_num_threads(4)
HOURS_PER_BATCH = 16
Y_SCALE = 20.0

fold = folds[0]
edges = build_edges(fold.train_start, fold.val_start)
full_ei = torch.from_numpy(edge_index(edges, pnode_ids)).long()
ei = full_ei
n_nodes = len(pnode_ids)


@lru_cache
def batch_edges(hours):
    return torch.cat([ei + i * n_nodes for i in range(hours)], dim=1)

In [ ]:
def feature_names(feature_set):
    core4 = FEATURES[:4]
    history = FEATURES + SYSTEM_FEATURES
    return {"core4": core4, "history": history, "all": history + ["area_forecast"] + WEATHER_FEATURES}[feature_set]


def node_inputs(block, feature_set):
    x = block["x"]
    if feature_set == "core4":
        return np.ascontiguousarray(x[..., :4])
    hours, nodes = x.shape[:2]
    parts = [x, np.broadcast_to(block["system"][:, None, :], (hours, nodes, block["system"].shape[1]))]
    if feature_set == "all":
        zone = data["node_weather_zone"]
        weather = block["weather"][:, np.maximum(zone, 0)]
        weather[:, zone < 0] = 0
        parts += [block["area"][:, data["node_area"]][..., None], weather]
    return np.concatenate(parts, axis=-1)


def build_tensors(feature_set):
    return {
        name: (
            torch.from_numpy(node_inputs(b, feature_set)),
            torch.from_numpy(np.nan_to_num(b["y"]) / Y_SCALE),
            torch.from_numpy(b["loss_mask"]),
        )
        for name, b in blocks.items()
    }


def run_variant(feature_set_name, graph):
    global feature_set, tensors, ei
    feature_set = feature_set_name
    tensors = build_tensors(feature_set)
    ei = full_ei if graph == "full" else torch.empty(2, 0, dtype=torch.long)
    batch_edges.cache_clear()
    return train_gnn()

In [ ]:
class SpreadGNN(nn.Module):
    def __init__(self, n_nodes, n_in, hidden=64, node_dim=16, layers=2):
        super().__init__()
        self.node_emb = nn.Embedding(n_nodes, node_dim)
        self.inp = nn.Linear(n_in + node_dim, hidden)
        self.convs = nn.ModuleList(SAGEConv(hidden, hidden) for _ in range(layers))
        self.out = nn.Linear(hidden, 1)

    def forward(self, x, edges):
        hours, nodes, _ = x.shape
        h = self.inp(torch.cat([x, self.node_emb.weight.expand(hours, -1, -1)], dim=-1)).relu()
        h = h.reshape(hours * nodes, -1)
        for conv in self.convs:
            h = h + conv(h, edges).relu()
        return self.out(h).reshape(hours, nodes)


def run_epoch(model, split, optimizer):
    x, y, mask = tensors[split]
    order = torch.randperm(len(x))
    n_batches = len(x) // HOURS_PER_BATCH
    total = 0.0
    for b in range(n_batches):
        idx = order[b * HOURS_PER_BATCH : (b + 1) * HOURS_PER_BATCH]
        pred = model(x[idx], batch_edges(len(idx)))
        m = mask[idx]
        loss = nn.functional.huber_loss(pred[m], y[idx][m], delta=1.0)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
        if (b + 1) % 100 == 0:
            logger.info(f"  batch {b + 1}/{n_batches}, mean loss {total / (b + 1):.4f}")
    return total / n_batches


@torch.no_grad()
def predict(model, split):
    model.eval()
    x = tensors[split][0]
    out = [model(x[i : i + HOURS_PER_BATCH], batch_edges(len(x[i : i + HOURS_PER_BATCH]))) for i in range(0, len(x), HOURS_PER_BATCH)]
    model.train()
    return torch.cat(out).numpy() * Y_SCALE


def evaluate(pred, split):
    y = blocks[split]["y"]
    m = blocks[split]["loss_mask"]
    spike = m & (np.abs(y) >= np.quantile(np.abs(y[m]), 0.99))
    err = np.abs(pred - y)
    return {"mae": float(err[m].mean()), "spike_mae": float(err[spike].mean())}

In [ ]:
params = {
    "hidden": 64, "node_dim": 16, "layers": 2, "lr": 1e-3, "weight_decay": 1e-5,
    "hours_per_batch": HOURS_PER_BATCH, "max_epochs": 30, "patience": 5, "seed": 0,
    "huber_delta": Y_SCALE, "y_scale": Y_SCALE,
}
mlflow.set_experiment("gridcast-lmp-spread")


def train_gnn():
    graph = "full" if ei.shape[1] else "none"
    n_features = tensors["train"][0].shape[-1]
    run_name = f"gnn_{feature_set}_{graph}_{pd.Timestamp.now():%Y%m%d}"
    kinds = edges["kind"].value_counts().add_prefix("n_edges_").to_dict() if graph == "full" else {}
    torch.manual_seed(params["seed"])
    with mlflow.start_run(run_name=run_name) as run:
        out = PROCESSED_DATA_DIR / "runs" / run.info.run_id
        out.mkdir(parents=True)
        mlflow.set_tags({"fold": 0, "feature_set": feature_set, "graph": graph, "model": "sage_gnn", "local_artifacts": str(out)})
        mlflow.log_params({
            **params, **kinds,
            "train_start": fold.train_start, "val_start": fold.val_start,
            "test_start": fold.test_start, "test_end": fold.test_end,
            "features": ",".join(feature_names(feature_set)), "n_features": n_features, "n_nodes": n_nodes,
            "n_edges_undirected": ei.shape[1] // 2,
            "clip_percentiles": "0.1,99.9", "nan_fill": 0,
        })
        logger.info(f"{run_name}: {len(tensors['train'][0])} train hours, {n_features} features, {ei.shape[1] // 2} edges")

        model = SpreadGNN(n_nodes, n_features, params["hidden"], params["node_dim"], params["layers"])
        optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
        best_val, best_epoch, stale = float("inf"), 0, 0
        for epoch in range(params["max_epochs"]):
            start = time.time()
            train_loss = run_epoch(model, "train", optimizer)
            val = evaluate(predict(model, "val"), "val")
            seconds = time.time() - start
            mlflow.log_metrics(
                {"train_loss": train_loss, "val_mae": val["mae"], "val_spike_mae": val["spike_mae"], "epoch_seconds": seconds},
                step=epoch,
            )
            logger.info(f"epoch {epoch}: train_loss {train_loss:.4f}, val_mae {val['mae']:.2f}, val_spike_mae {val['spike_mae']:.1f}, {seconds:.0f}s")
            if val["mae"] < best_val:
                best_val, best_epoch, stale = val["mae"], epoch, 0
                torch.save(model.state_dict(), out / "model.pt")
            else:
                stale += 1
                if stale >= params["patience"]:
                    logger.info(f"early stop: no improvement for {stale} epochs")
                    break

        model.load_state_dict(torch.load(out / "model.pt"))
        test = evaluate(predict(model, "test"), "test")
        zero = evaluate(np.zeros_like(blocks["test"]["y"]), "test")
        mlflow.log_metrics({
            "best_epoch": best_epoch, "best_val_mae": best_val,
            "test_mae": test["mae"], "test_spike_mae": test["spike_mae"],
            "zero_mae": zero["mae"], "zero_spike_mae": zero["spike_mae"],
            **{f"loss_mask_coverage_{name}": float(b["loss_mask"].mean()) for name, b in blocks.items()},
        })
        np.savez(out / "scalers.npz", **{f"{k}_{n}": a for k, v in scalers.items() for n, a in zip(["lo", "hi", "mean", "std"], v)})
        logger.info(f"best epoch {best_epoch}, val_mae {best_val:.2f}; test_mae {test['mae']:.2f}, test_spike_mae {test['spike_mae']:.1f} (zero {zero['mae']:.2f} / {zero['spike_mae']:.1f})")
    return test

In [ ]:
run_variant("history", "full")